In [1]:
import os
import sys
import random

In [2]:
curr_dir = os.getcwd()
par_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(par_dir)
sys.path.append(proj_dir)

In [3]:
from code_mutation.mutation_functions import CodeMutator
from database import MongoDBHelper
from code_generation.code_generation_tester import CodeGenerationHumanEvalHelper
from code_inconsistency.code_inconsistency_tester import CodeInconsistencyHumanEvalHelper

In [4]:
# %%script false --no-raise-error

db = MongoDBHelper()
base_qns_db = db.client["Base_Questions_DB"]
question_database = base_qns_db['HumanEval_Open_Ended']

In [7]:
c = 0
f1 = 0
f2 = 0
mutate_failure = {}
f3 = 0
n_f = 0

skippers = set()

for i in range(question_database.count_documents({})):
    task_id = f"HumanEvalo{i}"
    if task_id in skippers:
        continue
    qn = question_database.find_one({"_id": task_id})

    complete_sol = qn['qn'] + "\n" + qn["canon_solution"]
    check = qn['check']
    examples = qn['examples']
    original_qn = qn['qn']
    
    input_metadata = CodeInconsistencyHumanEvalHelper.extract_input_metadata(examples = examples, qn = original_qn)
    example = random.choice(list(examples.keys()))
    func_name = CodeInconsistencyHumanEvalHelper.extract_func_name_from_example(example)    

    if "for" not in complete_sol:
        n_f += 1
        continue

    try:
        namespace = {}
        exec(complete_sol, namespace)
        exec(check, namespace)
        namespace['check'](namespace[func_name])

    except:
        print(f"{task_id} complete solution has issues")
        f1+=1
    
    mutator = CodeMutator()

    try: 
        mutated_code = CodeMutator.mutate_for_to_while(complete_sol, input_metadata=input_metadata)
        if mutated_code.strip() == complete_sol.strip():
            n_f += 1
            continue
    except Exception as e:
        print(f"{task_id}: Could not mutate the code due to the following error > {type(e),e}")
        mutate_failure[type(e)] = mutate_failure.get(type(e), 0)+1
        f2+=1
        continue    

    try:
        namespace = {}
        exec(mutated_code, namespace)
        exec(check, namespace)
        namespace['check'](namespace[func_name])
        c += 1

    except Exception as e:
        # print(mutated_code)
        print(f"{task_id} mutated solution has issues > {e}")
        f3 += 1
    except KeyboardInterrupt as e:
        print("###",task_id)
    pass

HumanEvalo5: Could not mutate the code due to the following error > (<class 'code_mutation.mutation_functions.MutationFailedError'>, MutationFailedError("Solution could not be mutated due to the following error: <class 'AttributeError'> > 'Subscript' object has no attribute 'args'"))
HumanEvalo24 mutated solution has issues > integer modulo by zero
HumanEvalo37: Could not mutate the code due to the following error > (<class 'code_mutation.mutation_functions.MutationFailedError'>, MutationFailedError("Solution could not be mutated due to the following error: <class 'AttributeError'> > 'Tuple' object has no attribute 'id'"))
HumanEvalo38: Could not mutate the code due to the following error > (<class 'code_mutation.mutation_functions.MutationFailedError'>, MutationFailedError("Solution could not be mutated due to the following error: <class 'TypeError'> > unhashable type: 'list'"))
HumanEvalo64: Could not mutate the code due to the following error > (<class 'code_mutation.mutation_functi

In [8]:
print(f"{c} test cases were mutated successfully.")
print(f"{f1} test cases failed as complete solution failed.")
print(f"{f2} test cases failed as mutator failed to mutate.")
for key in mutate_failure:
    print(f"    - {key}: {mutate_failure[key]} failures")
print(f"{f3} test cases failed as mutated code failed to pass check.")
print(f"{n_f} test cases contain no for loops for mutation.")
print(c + f1+f2+f3 + n_f +len(skippers)== question_database.count_documents({}))

80 test cases were mutated successfully.
0 test cases failed as complete solution failed.
11 test cases failed as mutator failed to mutate.
    - <class 'code_mutation.mutation_functions.MutationFailedError'>: 8 failures
    - <class 'TypeError'>: 2 failures
    - <class 'AttributeError'>: 1 failures
8 test cases failed as mutated code failed to pass check.
62 test cases contain no for loops for mutation.
True
